# Hyperspectral against multispectral (Python)

**Fullstack Remote Sensing** · code by **Rifky Nauval Hendrawan, Remote Sensing Specialist** · [Chapter: P6 · Hyperspectral and Multispectral: How Many Colours Are Enough?](https://rifkynhsp-byte.github.io/Flux-Pixel-Planet/en/p6-hyperspectral.html) · [Bab (Bahasa Indonesia)](https://rifkynhsp-byte.github.io/Flux-Pixel-Planet/id/p6-hyperspectral.html) · [Open the JavaScript in the Code Editor](https://code.earthengine.google.com/?scriptPath=users/rifkynauvalhsp/Fullstack-Remote-Sensing:ch00h_hyperspectral)

Run the cells from top to bottom. The first cell installs what Colab lacks, fetches the book's data and helpers, and signs you in to Earth Engine with your own Google Cloud project ([how to get one](https://developers.google.com/earth-engine/guides/access)).

*Jalankan sel dari atas ke bawah. Sel pertama memasang paket, mengambil data buku, dan masuk ke Earth Engine dengan proyek Google Cloud Anda sendiri.*

In [ ]:
# @title Setup: packages, book data, Earth Engine sign-in
!pip -q install earthengine-api geemap tabulate
!test -d /content/book || git clone -q --depth 1 --filter=blob:none --sparse https://github.com/rifkynhsp-byte/Flux-Pixel-Planet.git /content/book
!git -C /content/book sparse-checkout set tools scripts data   # only what the code needs, not the website
import sys, os; sys.path[:0] = ['/content/book/tools', '/content/book/scripts/py']; os.chdir('/content/book')
__file__ = '/content/book/scripts/py/ch00h_hyperspectral.py'   # the script finds the book's data relative to itself
PROJECT = "your-cloud-project"  # @param {type:"string"}
import ee
ee.Authenticate()
ee.Initialize(project=PROJECT)
_init = ee.Initialize; ee.Initialize = lambda *a, **k: None   # the script must not sign in again

PRINCIPLES P6 | Hyperspectral against multispectral.

    multispectral   a few wide bands placed in atmospheric windows
                    (Sentinel-2: 13 bands, 15 to 180 nm wide)
    hyperspectral   hundreds of narrow, contiguous bands
                    (EMIT: 285 bands, about 7.4 nm apart, 381 to 2493 nm)

A multispectral band is roughly the average of the hyperspectral bands that
fall inside it. Everything that happens within that average, a shoulder, a
dip, a shift, is lost.

In [ ]:
# Fullstack Remote Sensing | Code by Rifky Nauval Hendrawan, Remote Sensing Specialist
# MIT licence: free to use and adapt; please keep this credit line.

"""
PRINCIPLES P6 | Hyperspectral against multispectral.

    multispectral   a few wide bands placed in atmospheric windows
                    (Sentinel-2: 13 bands, 15 to 180 nm wide)
    hyperspectral   hundreds of narrow, contiguous bands
                    (EMIT: 285 bands, about 7.4 nm apart, 381 to 2493 nm)

A multispectral band is roughly the average of the hyperspectral bands that
fall inside it. Everything that happens within that average, a shoulder, a
dip, a shift, is lost.
"""

import io
import math
from pathlib import Path

import ee
import matplotlib.pyplot as plt
import requests
from matplotlib.patches import Rectangle
from PIL import Image
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# 1. EMIT surface reflectance: the clearest dry-season pass, 26 August 2025
# ---------------------------------------------------------------------------
aoi = ee.Geometry.Rectangle([107.30, -7.05, 107.75, -6.75], None, False)   # Saguling to Bandung
emit = ee.ImageCollection("NASA/EMIT/L2A/RFL").filterBounds(aoi)
WAVE = ee.List(emit.first().get("reflectance_wavelengths"))
RBANDS = [f"reflectance_{i}" for i in range(285)]

In [ ]:
def clear(img):
    bad = (img.select("cloud_flag").Or(img.select("cirrus_flag"))
           .Or(img.select("dilated_cloud_flag")))
    refl = img.select(RBANDS)
    return refl.updateMask(bad.Not()).updateMask(refl.select("reflectance_60").gt(0))


SCENE = "EMIT_L2A_RFL_001_20250826T041241_2523802_036"   # 75 % of the box clear
rfl = clear(emit.filter(ee.Filter.eq("system:index", SCENE)).first()).clip(aoi)

# Land cover from Dynamic World, most common label of the same years
dw = (ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterBounds(aoi)
      .filterDate("2025-06-01", "2025-10-01").select("label").mode())
CLASSES = {0: "Water", 1: "Trees", 4: "Crops", 6: "Built"}
COLOURS = {"Water": "#1f78b4", "Trees": "#1b7837", "Crops": "#d9a400", "Built": "#7f7f7f"}

In [ ]:
def class_spectra():
    """Mean EMIT spectrum of each land-cover class, one row per wavelength.

    Only pixels at least 90 m inside a class are used, so that a 60 m EMIT
    pixel is not half water and half shore.
    """
    wl = np.array(WAVE.getInfo())
    rows = {"wavelength_nm": wl}
    for code, name in CLASSES.items():
        core = dw.eq(code).focalMin(90, "circle", "meters")   # away from edges: pure pixels
        m = rfl.updateMask(core).reduceRegion(
            ee.Reducer.mean(), aoi, scale=60, maxPixels=1e9, tileScale=4).getInfo()
        rows[name] = [m.get(b) for b in RBANDS]
    df = pd.DataFrame(rows)
    # Deep water-vapour absorption: the atmosphere leaves nothing to measure
    gap = df.wavelength_nm.between(1320, 1460) | df.wavelength_nm.between(1770, 1970) \
        | (df.wavelength_nm > 2450)
    df.loc[gap, list(CLASSES.values())] = np.nan
    return df


# Band windows (centre, width in nm), simplified to rectangles
S2 = {"B1": (443, 20), "B2": (490, 65), "B3": (560, 35), "B4": (665, 30), "B5": (705, 15),
      "B6": (740, 15), "B7": (783, 20), "B8": (842, 115), "B8A": (865, 20), "B9": (945, 20),
      "B11": (1610, 90), "B12": (2190, 180)}
L8 = {"B1": (443, 16), "B2": (482, 60), "B3": (561, 57), "B4": (655, 37), "B5": (865, 28),
      "B6": (1609, 85), "B7": (2201, 187)}

In [ ]:
def resample(df, bands):
    """What a multispectral sensor would report: the mean inside each band."""
    out = []
    for name, (c, w) in bands.items():
        inside = df[df.wavelength_nm.between(c - w / 2, c + w / 2)]
        row = {"band": name, "centre_nm": c, "width_nm": w}
        for cls in CLASSES.values():
            row[cls] = inside[cls].mean()
        out.append(row)
    return pd.DataFrame(out)

In [ ]:
def plot_spectra(df):
    s2 = resample(df, S2)
    fig, ax = plt.subplots(figsize=(10, 4.6))
    for cls, col in COLOURS.items():
        ax.plot(df.wavelength_nm, df[cls], color=col, lw=1.6, label=cls)
        ax.plot(s2.centre_nm, s2[cls], "o", color=col, ms=6, mec="white", mew=0.8)
    for _, b in s2.iterrows():
        ax.axvspan(b.centre_nm - b.width_nm / 2, b.centre_nm + b.width_nm / 2,
                   color="#94a3b8", alpha=0.12, lw=0)
    ax.text(0.99, 0.95, "lines: EMIT, 285 bands; dots: the same spectra in Sentinel-2's 12 bands (grey)",
            transform=ax.transAxes, ha="right", fontsize=8, color="#475569")
    ax.set_xlabel("Wavelength (nm)")
    ax.text(0, 1.03, "Surface reflectance", transform=ax.transAxes, fontsize=9, color="#475569")
    ax.set_xlim(380, 2460)
    ax.set_ylim(0, None)
    ax.legend(frameon=False, fontsize=8, loc="upper right", bbox_to_anchor=(1.0, 0.9))
    ax.set_title("285 bands draw the whole curve; Sentinel-2's 12 keep its outline and lose the detail",
                 loc="left", fontsize=11, pad=22)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    return fig

In [ ]:
def red_edge_position(wl, refl):
    """Wavelength of the steepest rise between 680 and 750 nm."""
    sel = (wl >= 680) & (wl <= 750)
    d = np.gradient(refl[sel], wl[sel])
    return float(wl[sel][np.nanargmax(d)])

In [ ]:
def plot_detail(df):
    wl = df.wavelength_nm.to_numpy()
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4.2))
    # (a) red edge: trees against crops
    for cls in ("Trees", "Crops"):
        r = df[cls].to_numpy()
        sel = (wl >= 650) & (wl <= 800)
        a1.plot(wl[sel], r[sel], "-o", ms=2.5, color=COLOURS[cls], lw=1.4,
                label=f"{cls}: red edge at {red_edge_position(wl, r):.0f} nm")
    for b in ("B4", "B5", "B6", "B7"):
        c, w = S2[b]
        a1.axvspan(c - w / 2, c + w / 2, color="#94a3b8", alpha=0.18, lw=0)
        a1.text(c, 0.98, b, ha="center", va="top", fontsize=7, color="#475569",
                transform=a1.get_xaxis_transform())
    a1.set_xlabel("Wavelength (nm)")
    a1.legend(frameon=False, fontsize=8, loc="lower right")
    a1.set_title("a. Red edge: 20 EMIT bands, 4 Sentinel-2 bands", loc="left", fontsize=9.5)
    # (b) shortwave infrared beyond 2000 nm: one wide Sentinel-2 band
    for cls in ("Built", "Crops", "Trees"):
        r = df[cls].to_numpy()
        sel = (wl >= 2000) & (wl <= 2440)
        a2.plot(wl[sel], r[sel], "-", color=COLOURS[cls], lw=1.6, label=cls)
    c, w = S2["B12"]
    a2.axvspan(c - w / 2, c + w / 2, color="#94a3b8", alpha=0.18, lw=0)
    a2.text(c, 0.02, "Sentinel-2 B12: one number", ha="center", transform=a2.get_xaxis_transform(),
            fontsize=7.5, color="#475569")
    a2.set_xlabel("Wavelength (nm)")
    a2.legend(frameon=False, fontsize=8)
    a2.set_title("b. 2000 to 2440 nm: about 60 EMIT bands, 1 Sentinel-2 band", loc="left",
                 fontsize=9.5)
    for ax in (a1, a2):
        ax.set_ylabel("Surface reflectance", fontsize=8, color="#475569")
        for s in ("top", "right"):
            ax.spines[s].set_visible(False)
    fig.tight_layout()
    return fig

In [ ]:
def band_table(df):
    s2 = resample(df, S2)
    s2.insert(0, "sensor", "Sentinel-2")
    return s2.round(3)

In [ ]:
def sensors_table():
    return pd.DataFrame([
        ["EMIT (ISS)", "hyperspectral", 285, "381-2493", "about 7.4", 60, "2022-"],
        ["EnMAP", "hyperspectral", "about 250", "420-2450", "6.5 / 10", 30, "2022-"],
        ["PRISMA", "hyperspectral", 239, "400-2500", "about 12", 30, "2019-"],
        ["Hyperion (EO-1)", "hyperspectral", "242 (198 usable)", "357-2576", "about 10", 30, "2000-2017"],
        ["Sentinel-2 MSI", "multispectral", 13, "443-2190", "15-180", "10/20/60", "2015-"],
        ["Landsat 8/9 OLI", "multispectral", "9", "433-2294", "16-187", "30 (15 pan)", "2013-"],
    ], columns=["Sensor", "Type", "Bands", "Range (nm)", "Band width (nm)", "Pixel (m)", "Years"])


# ---------------------------------------------------------------------------
# 3. From the street to the pixel: four places seen from the ground and from orbit
# ---------------------------------------------------------------------------
# Street View panoramas found near each place; the camera looks at a point a
# few tens of metres away whose Dynamic World class matches the place.
SITES = [
    {"name": "Building, Gedung Merdeka, Bandung", "pano": (-6.92133, 107.60981), "heading": 300,
     "dist": 25, "sv_date": "May 2024"},
    {"name": "Paddy, Majalaya", "pano": (-7.03767, 107.74397), "heading": 180, "dist": 45,
     "sv_date": "April 2021"},
    {"name": "Lake, Situ Cileunca", "pano": (-7.19754, 107.54934), "heading": 320, "dist": 80,
     "sv_date": "December 2021"},
    {"name": "Forest, Tahura Juanda", "pano": (-6.85715, 107.63307), "heading": 60, "dist": 30,
     "sv_date": "November 2019"},
]
SV_DIR = Path(__file__).resolve().parents[2] / "images" / "real"
DW_NAMES = ["water", "trees", "grass", "flooded vegetation", "crops", "shrub and scrub", "built",
            "bare", "snow and ice"]
IJB_NAMES = ["Plantation", "Forest", "Agriculture", "Open land", "Water", "Mangrove", "Paddy",
             "Oil palm", "Sago", "Rubber"]

In [ ]:
def target_point(site):
    lat, lon = site["pano"]
    b = math.radians(site["heading"])
    dlat = site["dist"] * math.cos(b) / 6371000
    dlon = site["dist"] * math.sin(b) / (6371000 * math.cos(math.radians(lat)))
    return lat + math.degrees(dlat), lon + math.degrees(dlon)

In [ ]:
def street_view(site, i):
    """The Street View photo, fetched once with the author's key and kept with attribution."""
    out = SV_DIR / f"p6-streetview-{i}.jpg"
    key_file = Path.home() / ".config" / "qsgi" / "maps-api-key"
    if not out.exists() and key_file.exists():
        r = requests.get("https://maps.googleapis.com/maps/api/streetview",
                         params={"size": "640x400", "location": "%f,%f" % site["pano"],
                                 "heading": site["heading"], "fov": 70, "pitch": 0,
                                 "source": "outdoor", "key": key_file.read_text().strip()},
                         timeout=60)
        if r.headers.get("content-type", "").startswith("image/"):
            out.write_bytes(r.content)
    return Image.open(out) if out.exists() else None


S2B = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]
S2C = [490, 560, 665, 705, 740, 783, 842, 865, 1610, 2190]

In [ ]:
def ground_truth_table():
    """For each target pixel: Sentinel-2 reflectance, indices, Dynamic World and the 10-class model."""
    pts = ee.FeatureCollection([ee.Feature(ee.Geometry.Point(target_point(st)[::-1]), {"site": st["name"]})
                                for st in SITES])
    box = pts.geometry().buffer(2000).bounds()
    s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(box)
          .filterDate("2025-06-01", "2025-10-01").filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40))
          .map(lambda im: im.updateMask(im.select("SCL").remap([4, 5, 6, 11], [1, 1, 1, 1], 0)))
          .select(S2B).median().divide(10000))
    ndvi = s2.normalizedDifference(["B8", "B4"]).rename("NDVI")
    evi = s2.expression("2.5*(N-R)/(N+6*R-7.5*B+1)", {"N": s2.select("B8"), "R": s2.select("B4"),
                                                       "B": s2.select("B2")}).rename("EVI")
    dwc = (ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterBounds(box).filterDate("2025-06-01", "2025-10-01"))
    dw_label = dwc.select("label").mode().rename("dw")
    dw_prob = dwc.select(["water", "trees", "grass", "flooded_vegetation", "crops",
                                           "shrub_and_scrub", "built", "bare"]).mean()
    emb = (ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
           .filter(ee.Filter.calendarRange(2024, 2024, "year")).filterBounds(box).mosaic())
    dem = ee.Image("USGS/SRTMGL1_003").select("elevation")
    ijb = (ee.Image.cat([emb, dem, ee.Terrain.slope(dem)])
           .classify(ee.Classifier.load("projects/ee-rifkynauvalhsp2/assets/LULC_Classifier_Embeddings_Topo_rubber"))
           .rename("ijb"))
    stack = s2.addBands([ndvi, evi, dw_label, ijb]).addBands(dw_prob.max(ee.Image(0)).reduce(ee.Reducer.max()).rename("dw_p"))
    rows = stack.sampleRegions(pts, scale=10).getInfo()["features"]
    df = pd.DataFrame([r["properties"] for r in rows])
    df["dw"] = df.dw.astype(int).map(lambda v: DW_NAMES[v])
    df["ijb"] = df.ijb.astype(int).map(lambda v: IJB_NAMES[v])
    return df.set_index("site").loc[[st["name"] for st in SITES]].reset_index()

In [ ]:
def s2_chip(site):
    lat, lon = target_point(site)
    d = 0.0018                                         # about 200 m either side
    region = ee.Geometry.Rectangle([lon - d, lat - d, lon + d, lat + d], None, False)
    img = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(region)
           .filterDate("2025-06-01", "2025-10-01").filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40))
           .select(["B4", "B3", "B2"]).median())
    url = img.visualize(min=200, max=2000, gamma=1.2).getThumbURL(
        {"region": region, "dimensions": 400, "format": "png", "crs": "EPSG:4326"})
    return Image.open(io.BytesIO(requests.get(url, timeout=300).content)), (lon - d, lon + d, lat - d, lat + d)

In [ ]:
def plot_street_to_pixel(df):
    n = len(SITES)
    fig, axes = plt.subplots(n, 3, figsize=(13, 3.3 * n), gridspec_kw={"width_ratios": [1.6, 1, 1.5]})
    for i, site in enumerate(SITES):
        row = df.iloc[i]
        a0, a1, a2 = axes[i]
        sv = street_view(site, i)
        if sv is not None:
            a0.imshow(sv)
        a0.set_axis_off()
        a0.set_title(f"{site['name']}: Street View, {site['sv_date']}", loc="left", fontsize=9)
        chip, ext = s2_chip(site)
        a1.imshow(chip, extent=ext)
        tl = target_point(site)
        a1.plot(site["pano"][1], site["pano"][0], "^", color="#facc15", ms=8, mec="black", label="camera")
        a1.add_patch(Rectangle((tl[1] - 0.000045, tl[0] - 0.000045), 0.00009, 0.00009, fill=False,
                               ec="#ef4444", lw=1.8))
        a1.set_xticks([]); a1.set_yticks([])
        a1.set_title("Sentinel-2, 2025: camera ▲, pixel looked at □", loc="left", fontsize=8)
        refl = [row[b] for b in S2B]
        a2.plot(S2C, refl, "-o", color="#1d4ed8", ms=4)
        a2.set_ylim(0, 0.45); a2.set_xlim(400, 2300)
        a2.set_title(f"Dynamic World: {row.dw}   |   10-class model: {row.ijb}", loc="left", fontsize=8.5)
        a2.text(0.98, 0.95, f"NDVI {row.NDVI:.2f}\nEVI {row.EVI:.2f}", transform=a2.transAxes,
                ha="right", va="top", fontsize=9,
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#cbd5e1"))
        a2.set_ylabel("Reflectance", fontsize=8)
        if i == n - 1:
            a2.set_xlabel("Wavelength (nm)")
        for sp in ("top", "right"):
            a2.spines[sp].set_visible(False)
    fig.suptitle("What you see from the street, what the satellite sees, and what two classifiers call it",
                 x=0.01, ha="left", fontsize=11.5)
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    return fig


SPECTRA = class_spectra()
GROUND = ground_truth_table()
RGB = {"bands": ["reflectance_35", "reflectance_23", "reflectance_11"],
       "min": 0, "max": 0.15, "gamma": 1.3}

In [ ]:
def products():
    return [
        {"kind": "map", "name": "p6-emit-rgb", "image": rfl, "region": aoi, "vis": RGB,
         "legend": "EMIT true colour (641, 552, 463 nm)",
         "title": "Saguling to Bandung, EMIT, 26 Aug 2025", "source": "NASA EMIT L2A. GEE.",
         "caption": "EMIT reflectance in true colour: three of its 285 bands, from the clearest "
                    "dry-season pass over the area (26 August 2025, 60 m). Cloud is masked. "
                    "Saguling reservoir is on the left, Bandung on the right."},
        {"kind": "chart", "name": "p6-spectra", "data": SPECTRA, "plot": plot_spectra,
         "caption": "Mean EMIT spectrum of each Dynamic World class on 26 August 2025 (lines) and the same "
                    "spectra averaged into Sentinel-2 bands (dots). Gaps are the water-vapour "
                    "absorption bands, where too little light reaches the sensor."},
        {"kind": "chart", "name": "p6-detail", "data": SPECTRA, "plot": plot_detail,
         "caption": "Two places where the detail matters. (a) The red edge: its exact position "
                    "shifts with chlorophyll and canopy, and EMIT samples it every 7 nm. "
                    "(b) Beyond 2000 nm, where minerals, cellulose and moisture leave narrow "
                    "features, Sentinel-2 has one band 180 nm wide."},
        {"kind": "table", "name": "p6-s2-table", "data": SPECTRA, "transform": band_table,
         "caption": "The same class spectra as Sentinel-2 would report them: the mean "
                    "reflectance inside each band."},
        {"kind": "chart", "name": "p6-street-to-pixel", "data": GROUND, "plot": plot_street_to_pixel,
         "live": False,
         "caption": "Four places in and around Bandung. Left: Google Street View, looking at a point "
                    "a few tens of metres from the camera. Middle: the same place in Sentinel-2, "
                    "with the camera (yellow) and the 10 m pixel it looks at (red). Right: that "
                    "pixel's spectral signature in ten Sentinel-2 bands, its NDVI and EVI, and the "
                    "labels Dynamic World and the author's 10-class model give it. Street View "
                    "images © Google."},
        {"kind": "table", "name": "p6-street-table", "data": GROUND,
         "columns": ["site", "dw", "ijb", "NDVI", "EVI", "B4", "B8", "B11"],
         "floatfmt": ("", "", "", ".2f", ".2f", ".3f", ".3f", ".3f"),
         "caption": "The same four pixels as numbers: labels, indices and three key bands (red, "
                    "near infrared, shortwave infrared)."},
        {"kind": "table", "name": "p6-sensors", "data": sensors_table,
         "caption": "Hyperspectral and multispectral sensors. Band counts and ranges from the "
                    "mission documentation; widths rounded."},
    ]

### The chapter's maps, charts and tables
The same products the book shows, drawn here. Maps also open as an interactive layer.

In [ ]:
from colab_render import show_all
show_all(products())